# Kriging workflow

This notebook walks the full kriging pipeline on synthetic rain-gauge data:
**variogram → fit → krige → cross-validate**. Everything is a method on `Samples`, a `FeatureCollection` subclass.

In [ ]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from geostatista import Samples

rng = np.random.default_rng(0)
xy = rng.uniform(0.0, 100.0, (80, 2))
rain = np.sin(xy[:, 0] / 25.0) * np.cos(xy[:, 1] / 25.0) * 10.0 + 20.0
gdf = gpd.GeoDataFrame({"rain": rain}, geometry=[Point(x, y) for x, y in xy], crs="EPSG:32633")
samples = Samples(gdf)
samples.head()

## 1. Empirical variogram — look before you krige

In [ ]:
vg = samples.variogram("rain", n_lags=12)
vg.to_dataframe()

## 2. Fit a model — the (nugget, sill, range) triple kriging needs

In [ ]:
vg.fit(model="spherical")
print(vg)
print("gamma(0) =", float(vg.predict(0.0)), " gamma(30) =", round(float(vg.predict(30.0)), 3))

## 3. Krige onto a grid — a 2-band surface (estimate + variance)

In [ ]:
surface = samples.krige("rain", vg, cell_size=5.0, n_neighbors=24)
arr = np.asarray(surface.read_array())
print("bands:", surface.band_count, " epsg:", surface.epsg, " shape:", arr.shape)
print("estimate range:", round(arr[0].min(), 2), "->", round(arr[0].max(), 2))
print("variance >= 0 everywhere:", bool(np.all(arr[1] >= -1e-9)))

The variance band — the reason to prefer kriging over IDW — is available as a `Dataset`:

In [ ]:
variance = surface.variance
print("variance band count:", variance.band_count)

## 4. Validate honestly — leave-one-out cross-validation

In [ ]:
cv = samples.cross_validate("rain", vg, n_neighbors=None)
cv.attrs["summary"]